In [3]:
# Ensure Java is installed and JAVA_HOME is set
import os
# print(os.environ['JAVA_HOME'])

# Set JAVA_HOME if not already set (update the path below to your Java installation path if needed)
java_home = "/usr/lib/jvm/java-21-openjdk-amd64"  # Update this path if your Java is elsewhere
os.environ["JAVA_HOME"] = java_home

# Check if JAVA_HOME exists
if not os.path.exists(os.environ["JAVA_HOME"]):
	raise EnvironmentError(f"JAVA_HOME directory does not exist: {os.environ['JAVA_HOME']}. Please update the path to your Java installation.")

from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local").appName("Word2Vec").config("spark.driver.memory", "8g").config("spark.executor.memory", "8g").config("spark.driver.maxResultSize", "8g").config("spark.kryoserializer.buffer.max", "1g").getOrCreate()


25/09/01 21:19:58 WARN SparkContext: Another SparkContext is being constructed (or threw an exception in its constructor). This may indicate an error, since only one SparkContext should be running in this JVM (see SPARK-2243). The other SparkContext was created at:
org.apache.spark.api.java.JavaSparkContext.<init>(JavaSparkContext.scala:59)
java.base/jdk.internal.reflect.NativeConstructorAccessorImpl.newInstance0(Native Method)
java.base/jdk.internal.reflect.NativeConstructorAccessorImpl.newInstance(NativeConstructorAccessorImpl.java:75)
java.base/jdk.internal.reflect.DelegatingConstructorAccessorImpl.newInstance(DelegatingConstructorAccessorImpl.java:53)
java.base/java.lang.reflect.Constructor.newInstanceWithCaller(Constructor.java:502)
java.base/java.lang.reflect.Constructor.newInstance(Constructor.java:486)
py4j.reflection.MethodInvoker.invoke(MethodInvoker.java:247)
py4j.reflection.ReflectionEngine.invoke(ReflectionEngine.java:374)
py4j.Gateway.invoke(Gateway.java:238)
py4j.command

In [2]:
spark_df=spark.read.csv('../data/matches.csv', header=True, inferSchema=True)

In [3]:
spark_df.toPandas().head(5)  # Display the first 5 rows of the DataFrame
spark_df.select('city').show(5)  # Display the first 5 rows of the 'city' column

+----------+
|      city|
+----------+
| Bangalore|
|Chandigarh|
|     Delhi|
|    Mumbai|
|   Kolkata|
+----------+
only showing top 5 rows


In [4]:
text_corpus=spark.read.text('/media/vivek/WD_HD/Machine_Learning/MachineLearning2018/DataCorpus/research_and_wiki_data/English/Wikipedia_data/wiki_en.txt')

In [5]:
text_corpus.show(5)  # Display the first 5 rows of the text corpus DataFrame

+--------------------+
|               value|
+--------------------+
|anarchism is poli...|
|autism is disorde...|
|percentage of dif...|
|writing cursive f...|
|alabama is state ...|
+--------------------+
only showing top 5 rows


In [5]:
 # Display the first 5 rows of the text corpus DataFrame
text_corpus.select('value').show(5)  # Display the first 5 rows of the 'value' column in the text corpus

+--------------------+
|               value|
+--------------------+
|anarchism is poli...|
|autism is disorde...|
|percentage of dif...|
|writing cursive f...|
|alabama is state ...|
+--------------------+
only showing top 5 rows


In [8]:
text_corpus.summary().show()  # Display summary statistics of the text corpus DataFrame

+-------+----------------------+
|summary|                 value|
+-------+----------------------+
|  count|               3433503|
|   mean|                  NULL|
| stddev|                  NULL|
|    min|  aa aa ee tamil te...|
|    25%|                  NULL|
|    50%|                  NULL|
|    75%|                  NULL|
|    max|혜성 comet is the f...|
+-------+----------------------+



In [ ]:
from pyspark.ml.feature import Tokenizer,StopWordsRemover,RegexTokenizer

tokenizer = RegexTokenizer(inputCol="value", outputCol="words", pattern="\\W")
wordsData = tokenizer.transform(text_corpus)

remover = StopWordsRemover(inputCol="words", outputCol="filtered")
filteredData = remover.transform(wordsData)
filteredData.select("value", "words", "filtered").show(5, truncate=False)  # Display the first 5 rows of the filtered DataFrame

from pyspark.ml.feature import Word2Vec

word2Vec = Word2Vec(vectorSize=50, minCount=20, inputCol="filtered", outputCol="word2vec")
model = word2Vec.fit(filteredData)
result = model.transform(filteredData)
model.save("word2vec_model_wiki_12.5_gb")

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

Py4JJavaError: An error occurred while calling o118.fit.
: java.lang.OutOfMemoryError: Java heap space
	at org.apache.spark.mllib.feature.Word2Vec.doFit(Word2Vec.scala:361)
	at org.apache.spark.mllib.feature.Word2Vec.fit(Word2Vec.scala:325)
	at org.apache.spark.ml.feature.Word2Vec.fit(Word2Vec.scala:183)
	at org.apache.spark.ml.feature.Word2Vec.fit(Word2Vec.scala:122)
	at java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)
	at java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke(NativeMethodAccessorImpl.java:75)
	at java.base/jdk.internal.reflect.DelegatingMethodAccessorImpl.invoke(DelegatingMethodAccessorImpl.java:52)
	at java.base/java.lang.reflect.Method.invoke(Method.java:580)
	at py4j.reflection.MethodInvoker.invoke(MethodInvoker.java:244)
	at py4j.reflection.ReflectionEngine.invoke(ReflectionEngine.java:374)
	at py4j.Gateway.invoke(Gateway.java:282)
	at py4j.commands.AbstractCommand.invokeMethod(AbstractCommand.java:132)
	at py4j.commands.Call